In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import time
import os
from tqdm import tqdm

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely import total_bounds

import av2.rendering.vector as vector_plotting_utils
from av2.datasets.motion_forecasting.scenario_serialization import load_argoverse_scenario_parquet
from av2.map.map_api import ArgoverseStaticMap

from tracs.av2_loader import AV2StaticMapLoader, AV2AgentLoader
from promis import StaRMap, ProMis
from promis.geo import CartesianRasterBand

### A priori Analysis
Where are the highest possible gains? Look for the worst predictions for different baselines.

In [ ]:
data_root = Path("../data")
backbone = "DeMo"

submission_path = data_root / "av2_backbone_outputs" / backbone / "submission.parquet"
metrics_path = data_root / "av2_backbone_outputs" / backbone / "metrics.parquet"
gt_path = data_root / "av2" / "av2_mf_focal_test_annotations.parquet"

In [ ]:
def get_worst(metrics_path: str, n_head = 9):
    metrics_df = pd.read_parquet(metrics_path)
    print("before filtering:", len(metrics_df))
    metrics_df = metrics_df[(metrics_df["is_missed_prediction1"]) & (~metrics_df["is_missed_prediction6"])]
    print("after filtering:", len(metrics_df))
    return metrics_df.sort_values("minFDE1", ascending=False).head(n_head)

worst_of = get_worst(metrics_path)
worst_of[["scenario_id", "track_id", "minFDE1", "minFDE6", "brier_minFDE1", "brier_minFDE6"]]

In [ ]:
def plot_worst(gt_path, submission_path, worst_submissions):
    NROWS, NCOLS = 3, 3
    SID_DIGITS = 36

    predictions_df = pd.read_parquet(submission_path)
    gt_df = pd.read_parquet(gt_path)

    fig, axes = plt.subplots(nrows=NROWS, ncols=NCOLS, figsize=(12, 12))
    cmap = matplotlib.colormaps.get("coolwarm_r")

    for i, (_, row) in enumerate(worst_submissions.iterrows()):
        scenario_id = str(row["scenario_id"])
        scenario_id = "0"*(SID_DIGITS - len(scenario_id)) + scenario_id
        
        scenario_directory = data_root / "av2" / "test" / scenario_id
        av2_map = ArgoverseStaticMap.from_json(
            scenario_directory / f"log_map_archive_{scenario_id}.json")
        av2_scenario = load_argoverse_scenario_parquet(
            scenario_directory / f"scenario_{scenario_id}.parquet")
        
        ax = axes[i//NROWS, i%NCOLS]
        gt_traj = gt_df[gt_df["scenario_id"] == scenario_id].iloc[0]
        ax.plot(gt_traj["gt_trajectory_x"], gt_traj["gt_trajectory_y"], c="black", alpha = 0.9)

        # plot all lane segments
        for _, ls in av2_map.vector_lane_segments.items():
            vector_plotting_utils.draw_polygon_mpl(
                ax, ls.polygon_boundary, color="gray", linewidth=0.5
            )
            vector_plotting_utils.plot_polygon_patch_mpl(
                ls.polygon_boundary, ax, color="gray", alpha=0.2
            )

        # plot all pedestrian crossings
        for _, pc in av2_map.vector_pedestrian_crossings.items():
            vector_plotting_utils.draw_polygon_mpl(ax, pc.polygon, color="dimgray", linewidth=0.5)
            vector_plotting_utils.plot_polygon_patch_mpl(
                pc.polygon, ax, color="dimgray", alpha=0.2
            )
        
        # plot all agent tracks 
        for track in av2_scenario.tracks:
            track_color = "black" if track.track_id == row["track_id"] else "y"
            trajectory = np.array([[*state.position] for state in track.object_states])
            ax.plot(trajectory[:, 0], trajectory[:, 1], color=track_color, linewidth=0.5)

        local_predictions = predictions_df[predictions_df["scenario_id"] == row["scenario_id"]]
        for _, pred in local_predictions.iterrows():
            ax.plot(pred["predicted_trajectory_x"], pred["predicted_trajectory_y"], color=cmap(pred["probability"]), linewidth=1, alpha=0.9)
        
        # crop image
        x_start = local_predictions.iloc[0]["predicted_trajectory_x"][0]
        y_start = local_predictions.iloc[0]["predicted_trajectory_y"][0]
        xw, yw = 80, 80
        ax.set_xlim(x_start - xw, x_start + xw)
        ax.set_ylim(y_start - yw, y_start + yw)

    plt.show()

In [ ]:
plot_worst(gt_path, submission_path, worst_of)

In [ ]:
scenario_id = "03219b3e-b385-4779-99f5-aa1ca5783f33"

feature_desc = {
    "crossing": "PEDESTRIAN_CROSSING",
    "street": "DRIVABLE_AREA",
    "intersection": "INTERSECTION",
    "lane": "VEHICLE",
    "bike_lane": "BIKE",
    "bus_lane": "BUS",
    "solid_line": ["DOUBLE_SOLID_YELLOW", "DOUBLE_SOLID_WHITE", "SOLID_YELLOW", "SOLID_WHITE",],
    "hybrid_line": ["DASH_SOLID_YELLOW", "DASH_SOLID_WHITE", "SOLID_DASH_WHITE", "SOLID_DASH_YELLOW",],
    "dashed_line": ["DASHED_WHITE", "DASHED_YELLOW", "DOUBLE_DASH_YELLOW", "DOUBLE_DASH_WHITE",],
}

map_file_path = data_root / "av2" / "val" / scenario_id / f"log_map_archive_{scenario_id}.json"
av2_loader = AV2StaticMapLoader(map_file_path, feature_desc)
scenario_uam = av2_loader.to_cartesian_map()
scenario_uam.apply_covariance(0.6 * np.eye(2))

In [ ]:
# find area of interest
# build & shift rasterband to match that area
# initialize star map
# initialize promis

# then, try to update.
# build another class that parses geometries from argoverse to promis landscape

geoms = [f.geometry for all_f in scenario_uam.features.values()
         for f in all_f if f.location_type == "lane"]
xmin, ymin, xmax, ymax = tuple(total_bounds(geoms))
raster = CartesianRasterBand(
    origin=None,
    resolution=(100, 100),
    width=xmax - xmin,
    height=ymax - ymin,
)
raster.data["east"] += (xmin + xmax) / 2
raster.data["north"] += (ymin + ymax) / 2

resin_program = """
over(street) <- source("/star_map/over/street", Probability).
over(crossing) <- source("/star_map/over/crossing", Probability).
over(lane) <- source("/star_map/over/lane", Probability).

distance(intersection) <- source("/star_map/distance/intersection", Density).

landscape if over(street) and not over(crossing) and distance(intersection) < 37.
landscape -> target("/landscape").
"""

star_map = StaRMap(scenario_uam)
star_map.initialize(raster, 3, resin_program)

In [ ]:
star_map.get("distance", "intersection").parameters.scatter(plot_basemap=False)
pass

In [ ]:
promis = ProMis(star_map, resin_program, 100*100, True)
promis.initialize(raster)
landscape = promis.update()
im = landscape.scatter(plot_basemap=False, cmap="coolwarm_r")
plt.colorbar(im)
pass

In [ ]:
scenario_id = "00105804-8330-4cff-9806-740499abad92"
agent_loader = AV2AgentLoader(
    submission_path, split_path=data_root / "av2" / "test", scenario_id=scenario_id)
agent_loader.load()
dynamic_uam = agent_loader.to_cartesian_map()
dynamic_star = StaRMap(dynamic_uam)

In [ ]:
dynamic_logic = """
distance(vehicle) <- source("/star_map/distance/vehicle", Density).
speed <- source("/sensors/speed", Number).

# Define three speed levels for city driving with corresponding safe distances
low_speed if speed < 20.
medium_speed if speed > 20 and speed < 40.
high_speed if speed > 40.

# Safe distance requirements for each speed level
safe_at_low if low_speed and distance(vehicle) > 5.
safe_at_medium if medium_speed and distance(vehicle) > 10.
safe_at_high if high_speed and distance(vehicle) > 20.

# Landscape: safe distance must be maintained according to speed level
landscape if safe_at_low.
landscape if safe_at_medium.
landscape if safe_at_high.

landscape -> target("/landscape").
"""

test = """
speed <- source("/sensors/speed", Number).

landscape if speed < 10.

landscape -> target("/landscape").
"""


In [ ]:
geoms = [f.geometry for all_f in scenario_uam.features.values()
         for f in all_f if f.location_type == "lane"]
xmin, ymin, xmax, ymax = tuple(total_bounds(geoms))
raster = CartesianRasterBand(
    origin=None,
    resolution=(100, 100),
    width=xmax - xmin + 10,
    height=ymax - ymin + 10,
)
raster.data["east"] += (xmin + xmax) / 2
raster.data["north"] += (ymin + ymax) / 2

dynamic_star.initialize(raster, 5, dynamic_logic)

In [ ]:
dynamic_star.get("distance", "vehicle").parameters.scatter(plot_basemap=False)
pass

In [ ]:
promis = ProMis(dynamic_star, dynamic_logic, 100*100, True)
writer = promis._resin.make_writer_for("speed")
writer.write(np.array([10] * (100*100)), time.monotonic()) 
promis.initialize(raster)
landscape = promis.update()
im = landscape.scatter(plot_basemap=False, cmap="coolwarm_r")
plt.colorbar(im)
pass

In [ ]:
av2_test_dir = data_root / "av2" / "test"
focal_to_type = pd.DataFrame()
for s_id in tqdm(os.listdir(av2_test_dir)):
    setup_df = pd.read_parquet(av2_test_dir / s_id / f"scenario_{s_id}.parquet")
    focal_agent = setup_df[setup_df["focal_track_id"] == setup_df["track_id"]].iloc[0]

    row = {
        "scenario_id": s_id,
        "focal_track_id":  focal_agent["track_id"],
        "type": focal_agent["object_type"],
    }    
    focal_to_type = pd.concat([focal_to_type, pd.DataFrame(row, index=[0])], ignore_index=True)

In [ ]:
focal_to_type["type"].unique()
len(focal_to_type)

In [ ]:
test_star = StaRMap.load(data_root / "star_maps" / "6253725c-02d1-4de3-8d7c-9f7b3b8ee109.pkl")
agent_loader = AV2AgentLoader(submission_path, data_root / "av2" /
                              "test", scenario_id="6253725c-02d1-4de3-8d7c-9f7b3b8ee109")

In [ ]:
test = test_star.get("over", "crosswalk")

In [ ]:
td = {f.location_type: f for f in test_star.uam.features}
location_types = test_star.uam.location_types
for location_type in location_types:
    print(location_type, len(test_star.uam.filter(location_type).features))

In [ ]:
x_pred_flat = np.array([f_pred["x_traj"] for f_pred in agent_loader.focal_predictions]).ravel()
y_pred_flat = np.array([f_pred["y_traj"] for f_pred in agent_loader.focal_predictions]).ravel()
xmin, xmax = x_pred_flat.min(), x_pred_flat.max()
ymin, ymax = y_pred_flat.min(), y_pred_flat.max()

raster = CartesianRasterBand(
    origin=None,
    resolution=(100, 100),
    width=xmax - xmin + 40,
    height=ymax - ymin + 40,
)   
raster.data["east"] += (xmin + xmax) / 2
raster.data["north"] += (ymin + ymax) / 2

test.parameters.into(raster).scatter(plot_basemap=False)